In [ ]:
import sys, time, importlib
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import src.config as config
from gold_standard import GOLD_STANDARD

from sentence_transformers import CrossEncoder
eval_judge = CrossEncoder("BAAI/bge-reranker-base")

def is_semantic_hit(query, chunk_text, threshold=0.5):
    """
    Evaluates if the CHUNK actually answers the QUERY.
    """
    if not chunk_text or not query:
        return False
    score = eval_judge.predict([[query, chunk_text]])[0]
    return score > threshold

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
EXPERIMENTS = [
    {"id": "exp_01_llama1b_512",  "llm": "llama3.2:1b",   "chunk": 512,  "db": "chroma_db_512"},
    {"id": "exp_02_llama1b_1024", "llm": "llama3.2:1b",   "chunk": 1024, "db": "chroma_db_1024"},
    {"id": "exp_03_llama3b_512",  "llm": "llama3.2:3b",   "chunk": 512,  "db": "chroma_db_512"},
    {"id": "exp_04_llama3b_1024", "llm": "llama3.2:3b",   "chunk": 1024, "db": "chroma_db_1024"},
    {"id": "exp_05_qwen4b_512",   "llm": "qwen3:4b",    "chunk": 512,  "db": "chroma_db_512"},
    {"id": "exp_06_qwen4b_1024",  "llm": "qwen3:4b",    "chunk": 1024, "db": "chroma_db_1024"},
]

STRATEGIES = ["Sparse (BM25)", "Dense (Vector)", "Hybrid + Rerank", "HyDE"]

In [3]:
# --- DEBUG CELL ---
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd()))
import src.config as config
import importlib

# Setup for exp 01
config.LLM_MODEL = "llama3.2:1b"
config.CHUNK_SIZE = 512
config.CHROMA_DIR = Path.cwd() / "chroma_db_512"
config.TOP_K = 5

import src.retrievers, src.rag_chain
importlib.reload(src.retrievers)
importlib.reload(src.rag_chain)
from src.retrievers import get_retriever
from src.rag_chain import get_qa_chain
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=config.EMBEDDING_MODEL)
vs = Chroma(persist_directory=str(config.CHROMA_DIR), embedding_function=embeddings, collection_name="tuhh_courses")

retriever = get_retriever("Sparse (BM25)", vs)
chain = get_qa_chain(retriever)

test_query = "How many credit points is the Advanced Machine Learning module worth?"
print(f"Running: {test_query}\n")

try:
    resp = chain.invoke({'input': test_query})
    print("TYPE OF RESP:", type(resp))
    
    # Try to get docs (handle different chain formats)
    docs = resp.get('context', [])
    answer = resp.get('answer', 'NO ANSWER FOUND')
    
    print(f"NUMBER OF DOCS RETURNED: {len(docs)}")
    print(f"LLM ANSWER: {answer}\n")
    
    if len(docs) > 0:
        print(f"FIRST CHUNK PREVIEW: {docs[0].page_content[:200]}...")
    else:
        print("❌ ERROR: DOCS LIST IS EMPTY! The retriever failed.")
        
except Exception as e:
    print(f"❌ SCRIPT CRASHED: {e}")

 Initialising: Sparse (BM25)
Running: How many credit points is the Advanced Machine Learning module worth?

TYPE OF RESP: <class 'dict'>
NUMBER OF DOCS RETURNED: 5
LLM ANSWER: The Advanced Machine Learning module is worth 6 credit points.

FIRST CHUNK PREVIEW: Core Qualification (66 CP in total)

The core qualification contains the compulsory modules (each 6 CP):


Module Manual M.Sc. "Data Science"

Advanced Machine Learning
Linear Models and Regression
Bi...


In [11]:
all_master_rows = []

for exp in EXPERIMENTS:
    print(f"\n{'='*60}\nStarting: {exp['id']}\n{'='*60}")
    
    # 1. Setup folders and config overrides
    exp_dir = Path(f"results/{exp['id']}")
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    config.LLM_MODEL = exp['llm']
    config.CHUNK_SIZE = exp['chunk']
    config.CHROMA_DIR = Path.cwd() / exp['db']
    config.TOP_K = 10 # Fetch 10 for Recall@10
    
    # 2. Force Python to reload modules so they read the new config variables
    import src.retrievers, src.rag_chain
    importlib.reload(src.retrievers)
    importlib.reload(src.rag_chain)
    from src.retrievers import get_retriever
    from src.rag_chain import get_qa_chain
    
    # 3. Load Vector DB
    from langchain_chroma import Chroma
    from langchain_ollama import OllamaEmbeddings
    embeddings = OllamaEmbeddings(model=config.EMBEDDING_MODEL)
    vs = Chroma(persist_directory=str(config.CHROMA_DIR), embedding_function=embeddings, collection_name="tuhh_courses")
    
    exp_rows = []

    for strat in STRATEGIES:
        print(f"\n  -> Testing: {strat}")
        retriever = get_retriever(strat, vs)
        chain = get_qa_chain(retriever)
        
        for item in GOLD_STANDARD:
            t0 = time.time()
            try:
                resp = chain.invoke({'input': item['query']})
                docs = resp.get('context', [])
                generated_answer = resp.get('answer', 'CHAIN_ERROR')
            except Exception as e:
                docs, generated_answer = [], f"CHAIN_ERROR: {e}"
                print(f"  Chain crashed on {item['id']}: {e}")
            
            latency = round(time.time() - t0, 2)
            
            # 4. Calculate Metrics using Cross-Encoder
            # Safely calculate metrics
            acc_1, rec_5, rec_10, mrr_val = 0, 0, 0, 0.0
            
            if len(docs) > 0:
                # Pass the ITEM'S QUERY and the CHUNK TEXT
                pairs = [[item['query'], d.page_content] for d in docs[:10]]
                scores = eval_judge.predict(pairs)
                
                # Debug print for the very first query
                if item['id'] == "Q01":
                    print(f"      [Debug Q01] Scores: {[round(s,2) for s in scores[:3]]}")
                
                # Score > 0.0 means the chunk answers the query
                if scores[0] > 0.5: acc_1 = 1
                if any(s > 0.5 for s in scores[:5]): rec_5 = 1
                if any(s > 0.5 for s in scores[:10]): rec_10 = 1
                
                for rank, score in enumerate(scores, start=1):
                    if score > 0.5:
                        mrr_val = 1.0 / rank
                        break
            
            # 5. Extract text for qualitative analysis (replace newlines so CSV formats cleanly)
            top_chunk = docs[0].page_content.replace("\n", " | ")[:500] if len(docs) > 0 else "NO_DOCS_RETRIEVED"
            hyde_text = docs[0].metadata.get("hyde_hypothetical", "N/A").replace("\n", " | ") if len(docs) > 0 else "N/A"
            
            row = {
                "strategy": strat,
                "query_id": item['id'],
                "difficulty": item['difficulty'],
                "query": item['query'],
                "gold_answer": item['gold_answer'],
                "Accuracy@1": acc_1,
                "Recall@5": rec_5,
                "Recall@10": rec_10,
                "MRR": round(mrr_val, 3),
                "Latency_s": latency,
                "generated_answer": generated_answer.replace("\n", " | "),
                "retrieved_top_chunk": top_chunk,
                "hyde_hypothetical": hyde_text
            }
            exp_rows.append(row)
            all_master_rows.append(row)
            
    # 6. Save Individual CSV for this configuration
    exp_df = pd.DataFrame(exp_rows)
    exp_df.to_csv(exp_dir / "results.csv", index=False)
    print(f"\n  ✅ Saved detailed results to {exp_dir}/results.csv")

print("\n ALL 6 EXPERIMENTS FINISHED.")


Starting: exp_05_qwen4b_512

  -> Testing: Sparse (BM25)
 Initialising: Sparse (BM25)
      [Debug Q01] Scores: [np.float32(0.33), np.float32(0.07), np.float32(0.89)]

  -> Testing: Dense (Vector)
 Initialising: Dense (Vector)
      [Debug Q01] Scores: [np.float32(0.9), np.float32(0.89), np.float32(0.93)]

  -> Testing: Hybrid + Rerank
 Initialising: Hybrid + Rerank
  Loading BGE reranker model (~270 MB) …


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Reranker ready.
      [Debug Q01] Scores: [np.float32(0.93), np.float32(0.9), np.float32(0.89)]

  -> Testing: HyDE
 Initialising: HyDE
      [Debug Q01] Scores: [np.float32(0.0), np.float32(0.0), np.float32(0.0)]

  ✅ Saved detailed results to results/exp_05_qwen4b_512/results.csv

Starting: exp_06_qwen4b_1024

  -> Testing: Sparse (BM25)
 Initialising: Sparse (BM25)
      [Debug Q01] Scores: [np.float32(0.03), np.float32(0.74), np.float32(0.12)]

  -> Testing: Dense (Vector)
 Initialising: Dense (Vector)
      [Debug Q01] Scores: [np.float32(0.94), np.float32(0.93), np.float32(0.95)]

  -> Testing: Hybrid + Rerank
 Initialising: Hybrid + Rerank
  Loading BGE reranker model (~270 MB) …


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Reranker ready.
      [Debug Q01] Scores: [np.float32(0.95), np.float32(0.94), np.float32(0.93)]

  -> Testing: HyDE
 Initialising: HyDE
      [Debug Q01] Scores: [np.float32(0.0), np.float32(0.94), np.float32(0.0)]

  ✅ Saved detailed results to results/exp_06_qwen4b_1024/results.csv

 ALL 6 EXPERIMENTS FINISHED.


In [13]:
import pandas as pd
import os

# Combine all 6 experiment CSVs into one Master DataFrame
all_files = [f"results/{exp['id']}/results.csv" for exp in EXPERIMENTS]
master_df = pd.concat([pd.read_csv(f) for f in all_files if os.path.exists(f)], ignore_index=True)

# Create clean labels for the plot legends
master_df['Config'] = master_df['query_id'].apply(lambda x: x.split('_llama')[0].replace('exp_', 'Exp '))
master_df['Strategy'] = master_df['strategy']

print(f"Combined {len(master_df)} rows from {len(all_files)} experiments.")

Combined 720 rows from 6 experiments.


In [16]:
import plotly.express as px
import plotly.graph_objects as go

os.makedirs("results/plots_combined", exist_ok=True)

# 1. Group by Config and Strategy to get averages
scatter_df = master_df.groupby(['Config', 'Strategy']).agg({
    'MRR': 'mean', 
    'Latency_s': 'mean'
}).reset_index()

# 2. Create Scatter Plot
fig = go.Figure()

# Define a professional color palette
colors = {'Sparse (BM25)': '#1f77b4', 'Dense (Vector)': '#ff7f0e', 
          'Hybrid + Rerank': '#2ca02c', 'HyDE': '#d62728'}
markers = {'Sparse (BM25)': 'circle', 'Dense (Vector)': 'square', 
           'Hybrid + Rerank': 'diamond', 'HyDE': 'cross'}

for strategy in scatter_df['Strategy'].unique():
    df_strat = scatter_df[scatter_df['Strategy'] == strategy]
    fig.add_trace(go.Scatter(
        x=df_strat['Latency_s'], 
        y=df_strat['MRR'],
        mode='markers+text', 
        name=strategy,
        marker=dict(color=colors[strategy], size=12, symbol=markers[strategy], line=dict(width=1, color='black')),
        text=df_strat['Config'],
        textposition="top center",
        textfont=dict(size=9)
    ))

#Average Latency (Seconds) — Lower is Better ➡️",
#"Mean Reciprocal Rank (MRR) — Higher is Better ⬆️",
fig.update_layout(
    title="RAG Strategy Performance: Precision (MRR) vs. Computational Cost (Latency)",
    xaxis_title="Average Latency (Seconds)",
    yaxis_title="Mean Reciprocal Rank (MRR)",
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=12),
    legend_title_text="Retrieval Strategy",
    height=600,
    margin=dict(l=50, r=50, t=80, b=50)
)

# Add a subtle annotation explaining the axes
fig.add_annotation(x=0.5, y=-0.15, xref='paper', yref='paper',
                   text="Note: Top-Left quadrant represents the theoretical 'Optimal Zone' (Fast & Precise).",
                   showarrow=False, font=dict(size=10, color="gray"))

fig.write_html("results/plots_combined/01_pareto_frontier.html")
fig.show()

In [ ]:
import plotly.express as px

# Group by Strategy and calculate means across ALL configurations
heatmap_df = master_df.groupby('Strategy')[['Accuracy@1', 'Recall@5', 'Recall@10', 'MRR']].mean().reset_index()

# Melt the dataframe to format it for Plotly heatmap
heatmap_melted = heatmap_df.melt(id_vars='Strategy', 
                                  value_vars=['Accuracy@1', 'Recall@5', 'Recall@10', 'MRR'],
                                  var_name='Metric', value_name='Score')

# Pivot to create matrix
heatmap_matrix = heatmap_melted.pivot(index='Strategy', columns='Metric', values='Score')

fig = px.imshow(
    heatmap_matrix, 
    text_auto='.2f', # Show numbers inside the squares
    color_continuous_scale='RdYlGn', # Red (0.0) to Green (1.0)
    aspect="auto",
    title="Aggregate Strategy Profile (Averaged across all LLMs & Chunk Sizes)"
)

fig.update_layout(
    xaxis_title="Evaluation Metric",
    yaxis_title="Retrieval Strategy",
    font=dict(family="Arial, sans-serif", size=13),
    height=500,
    margin=dict(l=150, r=50, t=80, b=50)
)

fig.write_html("results/plots_combined/02_strategy_profile_heatmap.html")
fig.show()

In [21]:
import plotly.graph_objects as go

dropoff_df = master_df.groupby('Strategy')[['Accuracy@1', 'Recall@10']].mean().reset_index()

fig = go.Figure()

metrics = ['Accuracy@1', 'Recall@10']
x = dropoff_df['Strategy']

for metric in metrics:
    fig.add_trace(go.Bar(
        x=x,
        y=dropoff_df[metric],
        name=metric,
        marker_color='#2ca02c' if metric == 'Recall@10' else '#d62728',
        text=[f'{val:.2f}' for val in dropoff_df[metric]],
        textposition='auto'
    ))

fig.update_layout(
    barmode='group',
    title="The Ranking Penalty: Accuracy@1 vs Recall@10",
    xaxis_title="Retrieval Strategy",
    yaxis_title="Score (0.0 to 1.0)",
    yaxis=dict(range=[0, 1.1]),
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=12),
    legend_title_text="Metric",
    height=500
)

fig.add_annotation(x=0.5, y=-0.2, xref='paper', yref='paper',
                   text="Gap indicates correct chunk was found, but ranked too low for the LLM to use effectively.",
                   showarrow=False, font=dict(size=10, color="gray", style='italic'))

fig.write_html("results/plots_combined/03_ranking_dropoff.html")
fig.show()

In [29]:
import plotly.express as px

print("Generating individual experiment plots...\n")

for exp in EXPERIMENTS:
    exp_dir = f"results/{exp['id']}"
    plot_dir = f"{exp_dir}/plots"
    os.makedirs(plot_dir, exist_ok=True)
    
    csv_path = f"{exp_dir}/results.csv"
    if not os.path.exists(csv_path):
        continue
        
    df = pd.read_csv(csv_path)
    avg_df = df.groupby('strategy')[['Accuracy@1', 'Recall@5', 'MRR']].mean().reset_index()
    
    # Clean title
    title_str = exp['id'].replace("exp_", "").replace("_", " | ").replace("llama", "Llama ").replace("qwen", "Qwen ").replace("b", "B")
    
    fig = px.bar(
        avg_df, 
        x="strategy", 
        y=["Accuracy@1", "Recall@5", "MRR"],
        barmode="group",
        title=f"Retrieval Performance: {title_str}",
        text_auto='.2f',
        color_discrete_sequence=["#d62728", "#ff7f0e", "#1f77b4"]
    )
    
    fig.update_layout(
        yaxis_title="Score",
        yaxis=dict(range=[0, 1.1]),
        template="plotly_white",
        font=dict(family="Arial, sans-serif", size=11),
        legend_title="Metric",
        height=450,
        margin=dict(l=50, r=20, t=60, b=100),
        xaxis=dict(tickangle=-15) # Angle labels so long strategy names don't overlap
    )
    
    fig.write_html(f"{plot_dir}/individual_summary.html")
    fig.show()
    print(f" Saved: {plot_dir}/individual_summary.html")

print("\n All plots generated successfully.")

Generating individual experiment plots...



 Saved: results/exp_01_llama1b_512/plots/individual_summary.html


 Saved: results/exp_02_llama1b_1024/plots/individual_summary.html


 Saved: results/exp_03_llama3b_512/plots/individual_summary.html


 Saved: results/exp_04_llama3b_1024/plots/individual_summary.html


 Saved: results/exp_05_qwen4b_512/plots/individual_summary.html


 Saved: results/exp_06_qwen4b_1024/plots/individual_summary.html

 All plots generated successfully.


## Qualitative Analysis

In [ ]:
import pandas as pd
def calculate_rank(mrr_val):
    if pd.isna(mrr_val) or mrr_val <= 0:
        return 0
    return int(1 / mrr_val)

master_df['Exact_Rank'] = master_df['MRR'].apply(calculate_rank)

print("Calculated exact ranks for all queries.")

# 2. ANALYSIS 1: BM25 was right (@1), but Dense failed
print("\n" + "="*80)
print("SCENARIO 1: BM25 found it at Rank 1, but Dense FAILED to find it in Top 10")
print("="*80)

bm25_right_dense_wrong = master_df[
    (master_df['strategy'] == 'Sparse (BM25)') & (master_df['Exact_Rank'] == 1)
]

if not bm25_right_dense_wrong.empty:
    for idx, row in bm25_right_dense_wrong.iterrows():
        dense_row = master_df[(master_df['query_id'] == row['query_id']) & (master_df['strategy'] == 'Dense (Vector)')]
        if not dense_row.empty and dense_row.iloc[0]['Exact_Rank'] == 0:
            print(f"\nQuery: {row['query']}")
            print(f"BM25 Retrieved: {str(row['retrieved_top_chunk'])[:150]}...")
            print(f"Dense Retrieved: {str(dense_row.iloc[0]['retrieved_top_chunk'])[:150]}...")
            print("-"*40)
else:
    print("No instances found where BM25 perfectly succeeded and Dense completely failed.")

# 3. ANALYSIS 2: Dense/Hybrid was right (@1), but BM25 failed
print("\n" + "="*80)
print("SCENARIO 2: Dense/Hybrid found it at Rank 1, but BM25 FAILED")
print("="*80)

for idx, row in master_df[master_df['strategy'] == 'Sparse (BM25)'].iterrows():
    if row['Exact_Rank'] == 0:
        smart_strats = master_df[
            (master_df['query_id'] == row['query_id']) & 
            (master_df['strategy'].isin(['Dense (Vector)', 'Hybrid + Rerank'])) & 
            (master_df['Exact_Rank'] == 1)
        ]
        if not smart_strats.empty:
            print(f"\nQuery: {row['query']}")
            print(f"BM25 Retrieved: {str(row['retrieved_top_chunk'])[:150]}...")
            smart_row = smart_strats.iloc[0]
            print(f"{smart_row['strategy']} Retrieved: {str(smart_row['retrieved_top_chunk'])[:150]}...")
            print("-"*40)
            
# 4. ANALYSIS 3: Disagreements among Embedding Models
print("\n" + "="*80)
print("SCENARIO 3: Dense found it, but HyDE failed (Embedding vs Hallucination)")
print("="*80)

dense_right = master_df[
    (master_df['strategy'] == 'Dense (Vector)') & (master_df['Exact_Rank'] > 0)
]

if not dense_right.empty:
    for idx, row in dense_right.iterrows():
        hyde_row = master_df[(master_df['query_id'] == row['query_id']) & (master_df['strategy'] == 'HyDE')]
        if not hyde_row.empty and hyde_row.iloc[0]['Exact_Rank'] == 0:
            print(f"\nQuery: {row['query']}")
            # Using str() ensures no weird float/string errors if there are missing values
            print(f"HyDE Hallucination: {str(hyde_row.iloc[0]['hyde_hypothetical'])[:200]}...")
            print("-"*40)

✅ Calculated exact ranks for all queries.

SCENARIO 1: BM25 found it at Rank 1, but Dense FAILED to find it in Top 10

Query: Is Advanced Machine Learning compulsory for Data Science students?
BM25 Retrieved: Core Qualification (66 CP in total) |  | The core qualification contains the compulsory modules (each 6 CP): |  |  | Module Manual M.Sc. "Data Science...
Dense Retrieved: Program: Masters Data Sci | Module: M1552 - Advanced Machine Learning | ECTS: 3 |  | Module M1552: Advanced Machine Learning |  | Courses |  | Title |...
----------------------------------------

Query: Is Advanced Machine Learning compulsory for Data Science students?
BM25 Retrieved: Core Qualification (66 CP in total) |  | The core qualification contains the compulsory modules (each 6 CP): |  |  | Module Manual M.Sc. "Data Science...
Dense Retrieved: Program: Masters Data Sci | Module: M1552 - Advanced Machine Learning | ECTS: 3 |  | Module M1552: Advanced Machine Learning |  | Courses |  | Title |...
---------